<a href="https://colab.research.google.com/github/seshuy41/daily_complex_sql/blob/main/day1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark

In [2]:
!git clone https://github.com/seshuy41/daily_complex_sql.git

Cloning into 'daily_complex_sql'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 6 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), 4.14 KiB | 4.14 MiB/s, done.


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySparkPractice") \
    .master("local[*]") \
    .getOrCreate()

In [4]:
print(spark.version)

4.0.4


In [5]:
from pyspark.sql import SparkSession


data = [
    ("C001", "Ravi", "B1"),
    ("C002", "John", "M1"),
    ("C003", "Sarah", "L1"),
    ("C004", "David", "B2")
]

columns = ["customer_id", "customer_name", "postcode"]

customer_df = spark.createDataFrame(data, columns)

customer_df.show()

+-----------+-------------+--------+
|customer_id|customer_name|postcode|
+-----------+-------------+--------+
|       C001|         Ravi|      B1|
|       C002|         John|      M1|
|       C003|        Sarah|      L1|
|       C004|        David|      B2|
+-----------+-------------+--------+



In [6]:
customer_df.write.mode("overwrite").parquet("/content/customers")

In [7]:
df_1 = spark.read.parquet("/content/customers")
df_2 = df_1.select("customer_id", "customer_name")
df_2.show()
#

+-----------+-------------+
|customer_id|customer_name|
+-----------+-------------+
|       C003|        Sarah|
|       C004|        David|
|       C001|         Ravi|
|       C002|         John|
+-----------+-------------+



In [8]:
data = [
    ("O1001", "C001", "S01", "2026-01-10", "Completed", "2026-01-10 10:00"),
    ("O1002", "C001", "S02", "2026-01-15", "Completed", "2026-01-15 11:00"),
    ("O1002", "C001", "S02", "2026-01-15", "Completed", "2026-01-16 09:00"),
    ("O1003", "C002", "S01", "2026-01-20", "Cancelled", "2026-01-20 12:00"),
    ("O1004", "C002", "S01", "2026-02-05", "Completed", "2026-02-05 14:00"),
    ("O1005", "C003", "S03", "2026-02-10", "Completed", "2026-02-10 15:00")
]

columns = [
    "order_id",
    "customer_id",
    "store_id",
    "order_date",
    "order_status",
    "updated_at"
]

orders_df = spark.createDataFrame(data, columns)

orders_df.show(truncate=False)

+--------+-----------+--------+----------+------------+----------------+
|order_id|customer_id|store_id|order_date|order_status|updated_at      |
+--------+-----------+--------+----------+------------+----------------+
|O1001   |C001       |S01     |2026-01-10|Completed   |2026-01-10 10:00|
|O1002   |C001       |S02     |2026-01-15|Completed   |2026-01-15 11:00|
|O1002   |C001       |S02     |2026-01-15|Completed   |2026-01-16 09:00|
|O1003   |C002       |S01     |2026-01-20|Cancelled   |2026-01-20 12:00|
|O1004   |C002       |S01     |2026-02-05|Completed   |2026-02-05 14:00|
|O1005   |C003       |S03     |2026-02-10|Completed   |2026-02-10 15:00|
+--------+-----------+--------+----------+------------+----------------+



In [9]:
orders_df.write.mode("overwrite").parquet("/content/orders")

In [10]:
data = [
    ("O1001", "P01", 2, 50),
    ("O1001", "P02", 1, 100),
    ("O1002", "P01", 1, 50),
    ("O1002", "P03", 2, 75),
    ("O1004", "P02", 2, 100),
    ("O1005", "P04", 1, 500),
    ("O1005", "P05", 2, 150)
]

columns = [
    "order_id",
    "product_id",
    "quantity",
    "unit_price"
]

order_items_df = spark.createDataFrame(data, columns)
order_items_df.write.mode("overwrite").parquet("/content/order_items")
order_items_df.show(truncate=False)

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
|O1001   |P01       |2       |50        |
|O1001   |P02       |1       |100       |
|O1002   |P01       |1       |50        |
|O1002   |P03       |2       |75        |
|O1004   |P02       |2       |100       |
|O1005   |P04       |1       |500       |
|O1005   |P05       |2       |150       |
+--------+----------+--------+----------+



In [11]:
columns_return = [
    "return_id",
    "order_id",
    "product_id",
    "return_quantity",
    "return_data"
]
data_return= [
    ("R001","O1001","P01",1,"2026-01-20"),
    ("R002","O1002","P03",1,"2026-01-25"),
    ("R003","O1005","P05",1,"2026-02-20"),

]

return_df= spark.createDataFrame(data_return,columns_return)
return_df.show()
return_df.write.mode("overwrite").parquet("/content/returns")

+---------+--------+----------+---------------+-----------+
|return_id|order_id|product_id|return_quantity|return_data|
+---------+--------+----------+---------------+-----------+
|     R001|   O1001|       P01|              1| 2026-01-20|
|     R002|   O1002|       P03|              1| 2026-01-25|
|     R003|   O1005|       P05|              1| 2026-02-20|
+---------+--------+----------+---------------+-----------+



In [12]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col , row_number ,count

In [13]:
w = Window.partitionBy(col("order_id")).orderBy(col("updated_at").desc())

In [14]:
df_rm_dup =(orders_df
            .withColumn("rn",row_number().over(w))
            .filter(col("rn")==1)


)

df_rm_dup.show()

+--------+-----------+--------+----------+------------+----------------+---+
|order_id|customer_id|store_id|order_date|order_status|      updated_at| rn|
+--------+-----------+--------+----------+------------+----------------+---+
|   O1001|       C001|     S01|2026-01-10|   Completed|2026-01-10 10:00|  1|
|   O1002|       C001|     S02|2026-01-15|   Completed|2026-01-16 09:00|  1|
|   O1003|       C002|     S01|2026-01-20|   Cancelled|2026-01-20 12:00|  1|
|   O1004|       C002|     S01|2026-02-05|   Completed|2026-02-05 14:00|  1|
|   O1005|       C003|     S03|2026-02-10|   Completed|2026-02-10 15:00|  1|
+--------+-----------+--------+----------+------------+----------------+---+



In [15]:
df_completed = (df_rm_dup.filter(col("order_status")=="Completed"))
df_completed.show()

+--------+-----------+--------+----------+------------+----------------+---+
|order_id|customer_id|store_id|order_date|order_status|      updated_at| rn|
+--------+-----------+--------+----------+------------+----------------+---+
|   O1001|       C001|     S01|2026-01-10|   Completed|2026-01-10 10:00|  1|
|   O1002|       C001|     S02|2026-01-15|   Completed|2026-01-16 09:00|  1|
|   O1004|       C002|     S01|2026-02-05|   Completed|2026-02-05 14:00|  1|
|   O1005|       C003|     S03|2026-02-10|   Completed|2026-02-10 15:00|  1|
+--------+-----------+--------+----------+------------+----------------+---+



In [16]:
df_total_order = df_rm_dup.groupBy(col("customer_id")).agg(count(col("order_id")).alias("total_orders"))
df_total_order.show()

+-----------+------------+
|customer_id|total_orders|
+-----------+------------+
|       C003|           1|
|       C001|           2|
|       C002|           2|
+-----------+------------+



In [17]:
df_com_order = df_completed.groupBy(col("customer_id")).agg(count(col("order_id")).alias("completed_orders"))
df_com_order.show()

+-----------+----------------+
|customer_id|completed_orders|
+-----------+----------------+
|       C003|               1|
|       C001|               2|
|       C002|               1|
+-----------+----------------+



In [29]:
from pyspark.sql.functions import sum
df_agg = order_items_df.groupBy(col("order_id")).agg(sum(col("quantity") * col("unit_price") ).alias("gross_sales"))
df_agg.show()

+--------+-----------+
|order_id|gross_sales|
+--------+-----------+
|   O1002|        200|
|   O1001|        200|
|   O1004|        200|
|   O1005|        800|
+--------+-----------+



In [21]:
order_items_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price: long (nullable = true)

